In [0]:
from pyspark.sql.functions import col, substring_index, regexp_replace, when
import os

In [0]:
table_path = "workspace.default.cloud_app_events_banking_dlp"
current_directory = os.getcwd() 
df = spark.table(table_path)

In [0]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = os.path.dirname(notebook_path)

#Get File Name and Extension 
- This Feature will be extracting file extension from file name. For DLP PDFs, CSVs, and Text files are higher risk than system files (like .exe) 
because they are the primary containers for sensitive client PII.

In [0]:
df_features = df.withColumn("file_name", regexp_replace(col("ObjectName"),r"\.[^.]+$", "")
                            ).withColumn("file_extension", substring_index(col("ObjectName"), ".", -1))
display(df_features.limit(100))


Action_ID,Timestamp,ActionType,ObjectName,Target_Domain,AccountDisplayName,Position,Risk_Label,file_name,file_extension
1,2026-02-12T03:32:03.000Z,FileUploaded,onboarding_checklist.docx,teams.microsoft.com,User_765,Software Developer,0,onboarding_checklist,docx
2,2026-01-29T13:21:54.000Z,FileUploaded,meeting_notes.docx,outlook.office365.com,User_325,Loan Officer,0,meeting_notes,docx
3,2026-02-17T12:56:41.000Z,FileUploaded,auth_service.py,linkedin.com,User_532,Software Developer,1,auth_service,py
4,2026-01-29T17:16:53.000Z,FileUploaded,onboarding_checklist.docx,outlook.office365.com,User_144,Relationship Banker,0,onboarding_checklist,docx
5,2026-02-02T08:20:27.000Z,FileUploaded,audit_log.pdf,workday.com,User_529,Relationship Banker,0,audit_log,pdf
6,2026-02-06T14:47:14.000Z,FileUploaded,budget_draft.xlsx,workday.com,User_144,System Architect,0,budget_draft,xlsx
7,2026-01-26T17:58:40.000Z,FileUploaded,annual_review.pdf,secure.bank.com,User_204,Loan Officer,0,annual_review,pdf
8,2026-02-02T04:02:31.000Z,FileUploaded,budget_draft.xlsx,workday.com,User_146,Relationship Banker,0,budget_draft,xlsx
9,2026-02-16T07:17:29.000Z,FileUploaded,compliance_v1.pdf,teams.microsoft.com,User_704,Wealth Manager,0,compliance_v1,pdf
10,2026-02-12T01:21:59.000Z,FileUploaded,training_material.pdf,internal.bank.sharepoint.com,User_338,Wealth Manager,0,training_material,pdf


#Double Extension Detection:
- Identifies "masked" files ('data.csv.zip'). This flags attempts 
to bypass file-type filters by hiding sensitive extensions inside 
another filename.


In [0]:
extension_pattern = r"\.(pdf|docx|xlsx|doc|xls|ppt|pptx|csv|zip|7z|rar|py|sh|bat|ps1|sql|txt)"

df_features = df_features.withColumn(
    "double_extension_check", 
    when(col("file_name").rlike(extension_pattern), 1).otherwise(0)
)

  
                                  
display(df_features.limit(100))

Action_ID,Timestamp,ActionType,ObjectName,Target_Domain,AccountDisplayName,Position,Risk_Label,file_name,file_extension,double_extension_check
1,2026-02-12T03:32:03.000Z,FileUploaded,onboarding_checklist.docx,teams.microsoft.com,User_765,Software Developer,0,onboarding_checklist,docx,0
2,2026-01-29T13:21:54.000Z,FileUploaded,meeting_notes.docx,outlook.office365.com,User_325,Loan Officer,0,meeting_notes,docx,0
3,2026-02-17T12:56:41.000Z,FileUploaded,auth_service.py,linkedin.com,User_532,Software Developer,1,auth_service,py,0
4,2026-01-29T17:16:53.000Z,FileUploaded,onboarding_checklist.docx,outlook.office365.com,User_144,Relationship Banker,0,onboarding_checklist,docx,0
5,2026-02-02T08:20:27.000Z,FileUploaded,audit_log.pdf,workday.com,User_529,Relationship Banker,0,audit_log,pdf,0
6,2026-02-06T14:47:14.000Z,FileUploaded,budget_draft.xlsx,workday.com,User_144,System Architect,0,budget_draft,xlsx,0
7,2026-01-26T17:58:40.000Z,FileUploaded,annual_review.pdf,secure.bank.com,User_204,Loan Officer,0,annual_review,pdf,0
8,2026-02-02T04:02:31.000Z,FileUploaded,budget_draft.xlsx,workday.com,User_146,Relationship Banker,0,budget_draft,xlsx,0
9,2026-02-16T07:17:29.000Z,FileUploaded,compliance_v1.pdf,teams.microsoft.com,User_704,Wealth Manager,0,compliance_v1,pdf,0
10,2026-02-12T01:21:59.000Z,FileUploaded,training_material.pdf,internal.bank.sharepoint.com,User_338,Wealth Manager,0,training_material,pdf,0


## Risky Domains Flag:

* Checks the **Target Domain** and flags it if the domain exists in `risky_domains.csv`. Since these domains are not used in a standard business process, there is a higher risk of a data leak associated with them.

In [0]:

csv_path = os.path.join(current_directory, "risky_domains.csv")
risky_domains_df = spark.read.csv(csv_path, header=True, inferSchema=True)


In [0]:
risky_domains_df = risky_domains_df.withColumn(
    "Domains", 
    func.explode(func.split(func.col("Domain"), r",\s*"))
)

risky_domains_list = [row[0] for row in risky_domains_df.select("Domains").collect()]

regex_pattern = "|".join([d.replace(".", r"\.") for d in risky_domains_list])

df_features = df_features.withColumn(
    "is_risky_destination",
    func.when(func.lower(func.col("Target_Domain")).rlike(regex_pattern), 1).otherwise(0)
)

display(df_features)

#Clean Code Cell
with comments


In [0]:
from pyspark.sql.functions import col, lower, regexp_replace, substring_index, when

# Define extension pattern to match common file extensions
extension_pattern = r"\.(pdf|docx|xlsx|doc|xls|ppt|pptx|csv|zip|7z|rar|py|sh|bat|ps1|sql|txt)"

# Flagging data exfiltration risks by matching target domains against a dynamic blacklist of non-business sanctioned apps.

risky_domains_df = risky_domains_df.withColumn(
    "Domains", 
    func.explode(func.split(func.col("Domain"), r",\s*"))
)

risky_domains_list = [row[0] for row in risky_domains_df.select("Domains").collect()]
regex_pattern = "|".join([d.replace(".", r"\.") for d in risky_domains_list])




df_features_clean = df_features.withColumn(
    # Extracts everything after the last dot
    "file_extension", lower(substring_index(col("ObjectName"), ".", -1))
).withColumn(
    # Extracts everything BEFORE the last dot by searching for the last dot and replacing whats after with an empty string
    "file_name", regexp_replace(col("ObjectName"), r"\.[^.]+$", "")
).withColumn(
    # Check 1: Is there an extension left after removing one?
    "double_extension_check", 
    when(col("file_name").rlike(extension_pattern), 1).otherwise(0)
).withColumn(
    # Check 2: Is the risky target domain?
    "is_risky_target_domain",
    when(func.lower(func.col("Target_Domain")).rlike(regex_pattern), 1).otherwise(0))
display(df_features_clean.limit(100))
   

Action_ID,Timestamp,ActionType,ObjectName,Target_Domain,AccountDisplayName,Position,Risk_Label,file_name,file_extension,double_extension_check
1,2026-02-12T03:32:03.000Z,FileUploaded,onboarding_checklist.docx,teams.microsoft.com,User_765,Software Developer,0,onboarding_checklist,docx,0
2,2026-01-29T13:21:54.000Z,FileUploaded,meeting_notes.docx,outlook.office365.com,User_325,Loan Officer,0,meeting_notes,docx,0
3,2026-02-17T12:56:41.000Z,FileUploaded,auth_service.py,linkedin.com,User_532,Software Developer,1,auth_service,py,0
4,2026-01-29T17:16:53.000Z,FileUploaded,onboarding_checklist.docx,outlook.office365.com,User_144,Relationship Banker,0,onboarding_checklist,docx,0
5,2026-02-02T08:20:27.000Z,FileUploaded,audit_log.pdf,workday.com,User_529,Relationship Banker,0,audit_log,pdf,0
6,2026-02-06T14:47:14.000Z,FileUploaded,budget_draft.xlsx,workday.com,User_144,System Architect,0,budget_draft,xlsx,0
7,2026-01-26T17:58:40.000Z,FileUploaded,annual_review.pdf,secure.bank.com,User_204,Loan Officer,0,annual_review,pdf,0
8,2026-02-02T04:02:31.000Z,FileUploaded,budget_draft.xlsx,workday.com,User_146,Relationship Banker,0,budget_draft,xlsx,0
9,2026-02-16T07:17:29.000Z,FileUploaded,compliance_v1.pdf,teams.microsoft.com,User_704,Wealth Manager,0,compliance_v1,pdf,0
10,2026-02-12T01:21:59.000Z,FileUploaded,training_material.pdf,internal.bank.sharepoint.com,User_338,Wealth Manager,0,training_material,pdf,0


#Tests Below

In [0]:
display(df_features.filter(col("double_extension_check") == 1))

Action_ID,Timestamp,ActionType,ObjectName,Target_Domain,AccountDisplayName,Position,Risk_Label,file_name,file_extension,double_extension_check
37,2026-01-31T19:16:39.000Z,FileUploaded,invoice.pdf.exe,ilovepdf.com,User_338,System Architect,1,invoice.pdf,exe,1
51,2026-02-12T18:17:03.000Z,FileUploaded,normal_file.docx.bat,chatgpt.com,User_242,Software Developer,1,normal_file.docx,bat,1
59,2026-02-23T09:23:31.000Z,FileUploaded,invoice.pdf.exe,chatgpt.com,User_854,Loan Officer,1,invoice.pdf,exe,1
69,2026-02-12T00:48:14.000Z,FileUploaded,normal_file.docx.bat,claude.ai,User_814,System Architect,1,normal_file.docx,bat,1
90,2026-02-15T20:05:11.000Z,FileUploaded,invoice.pdf.exe,claude.ai,User_263,Software Developer,1,invoice.pdf,exe,1
92,2026-02-02T13:33:35.000Z,FileUploaded,report.xlsx.zip,chatgpt.com,User_263,Wealth Manager,1,report.xlsx,zip,1
109,2026-02-11T01:42:48.000Z,FileUploaded,quarterly_data.csv.zip,facebook.com,User_370,Wealth Manager,1,quarterly_data.csv,zip,1
111,2026-01-25T20:58:45.000Z,FileUploaded,normal_file.docx.bat,linkedin.com,User_259,Relationship Banker,1,normal_file.docx,bat,1
122,2026-01-31T00:27:05.000Z,FileUploaded,quarterly_data.csv.zip,ilovepdf.com,User_384,Wealth Manager,1,quarterly_data.csv,zip,1
123,2026-02-09T04:38:30.000Z,FileUploaded,normal_file.docx.bat,wetransfer.com,User_381,Relationship Banker,1,normal_file.docx,bat,1


In [0]:
display(df_features.filter(df.ObjectName.contains(".pdf.")))

Action_ID,Timestamp,ActionType,ObjectName,Target_Domain,AccountDisplayName,Position,Risk_Label,file_name,file_extension,double_extension_check
37,2026-01-31T19:16:39.000Z,FileUploaded,invoice.pdf.exe,ilovepdf.com,User_338,System Architect,1,invoice.pdf,exe,1
59,2026-02-23T09:23:31.000Z,FileUploaded,invoice.pdf.exe,chatgpt.com,User_854,Loan Officer,1,invoice.pdf,exe,1
90,2026-02-15T20:05:11.000Z,FileUploaded,invoice.pdf.exe,claude.ai,User_263,Software Developer,1,invoice.pdf,exe,1
164,2026-02-12T06:05:48.000Z,FileUploaded,invoice.pdf.exe,wetransfer.com,User_195,System Architect,1,invoice.pdf,exe,1
181,2026-02-17T19:19:35.000Z,FileUploaded,invoice.pdf.exe,facebook.com,User_132,Relationship Banker,1,invoice.pdf,exe,1
225,2026-02-04T17:17:25.000Z,FileUploaded,invoice.pdf.exe,gemini.google.com,User_877,System Architect,1,invoice.pdf,exe,1
272,2026-02-17T22:05:05.000Z,FileUploaded,invoice.pdf.exe,dropbox.com,User_658,Wealth Manager,1,invoice.pdf,exe,1
292,2026-02-13T00:33:19.000Z,FileUploaded,invoice.pdf.exe,linkedin.com,User_467,Wealth Manager,1,invoice.pdf,exe,1
391,2026-02-18T04:03:45.000Z,FileUploaded,invoice.pdf.exe,dropbox.com,User_171,Software Developer,1,invoice.pdf,exe,1
392,2026-02-09T18:48:48.000Z,FileUploaded,invoice.pdf.exe,wetransfer.com,User_189,Relationship Banker,1,invoice.pdf,exe,1


In [0]:
df_features.select("Target_Domain").distinct().show()